In [3]:
import pandas as pd
import numpy as np

In [5]:
dfG = pd.read_excel('TimeSeries_FULL_GMonth.xlsx')
dfU = pd.read_excel('TimeSeries_FULL_UMonth.xlsx')


In [ ]:

Ueneral = dfU.copy()
# General

In [35]:
masterF = 'Real_Master_CPI.xlsx'
Real_master = pd.read_excel(masterF)
# Merge Real_master (RM)
RM = Real_master[['รหัส','ชุด', 'รายการ','กำหนดให้เก็บ','รายละเอียด']].rename(columns={'รหัส': 'CODE7'})
RM["CODE7"] = RM["CODE7"].astype(str).str.zfill(7)
# --- เพิ่มบรรทัดนี้เพื่อแก้ปัญหา TypeError ---
RM['กำหนดให้เก็บ'] = pd.to_numeric(RM['กำหนดให้เก็บ'], errors='coerce').fillna(0)

# ตอนนี้จะสามารถ Run บรรทัดนี้ได้แล้วครับ
RM = RM[(RM['ชุด']!='ชุด อ.') & (RM['กำหนดให้เก็บ'] > 0)]
RM = RM[['CODE7', 'รายการ']]
RM['CODE7'] = RM['CODE7'].astype(str)

RM

,CODE7,รายการ
5,1112104,แป้งทอดกรอบ
10,1112205,ขนมปังปอนด์
12,1112207,อาหารธัญพืช
15,1112210,ขนมอบ
80,1132001,นมสด
...,...,...
701,6311002,ค่าอาหารสำหรับตักบาตร
702,7100001,บุหรี่
704,7210001,เบียร์
705,7220001,ไวน์


In [ ]:
General = dfG.copy()
General = General[General['GROUP'] == 'อุตรดิตถ์']


General = General.rename(columns={"CODE_7_DIGITS":"CODE7"})
General['COMMODITY_CODE'] = General['COMMODITY_CODE'].astype(str).str.zfill(16)
General['DILLER_CODE'] = General['DILLER_CODE'].astype(str).str.zfill(10)
General["CODE7"] = General["CODE7"].astype(str).str.zfill(7)
General = pd.merge(RM,General,on='CODE7',how = 'left')
General = General.sort_values(by='CODE7').reset_index(drop=True)

General = General.rename(columns={'AVGtail_stable': 'ราคาคงที่กี่เดือน'})

# 3. ลบคอลัมน์ที่ไม่ต้องการออก (Drop)
cols_to_drop = [
    'REL6601', 'REL6602', 'REL6603', 'REL6604', 'REL6605', 'REL6606', 
    'REL6607', 'REL6608', 'REL6609', 'REL6610', 'REL6611', 'REL6612', 
    'REL6701', 'REL6702', 'REL6703', 'REL6704', 'REL6705', 'REL6706', 
    'REL6707', 'REL6708', 'REL6709', 'REL6710', 'REL6711', 'REL6712', 
    'REL6801', 'REL6802', 'REL6803', 'REL6804', 'REL6805', 'REL6806', 
    'REL6807', 'REL6808', 'REL6809', 'REL6810', 'REL6811', 'REL6812', 
    'REL6901', 'Rel_Max', 'Rel_Min', 'Relstreak_max_nonnull', 
    'Relstreak_max_eq100', 'Reltrail_nonnull', 'Reltrail_eq100', 
    'AVG_maxdiff', 'AVG_mindiff', 'CountAVG', 'AVG_Max', 'AVG_Min'
]

# ใช้ errors='ignore' เพื่อป้องกันกรณีบางคอลัมน์ไม่มีอยู่ใน DataFrame จะได้ไม่ error
General = General.drop(columns=cols_to_drop, errors='ignore')

output_file = "Steak_Month_ชุดทั่วไป.xlsx"
with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
    General.to_excel(writer, index=False, sheet_name='Data')
    
    workbook  = writer.book
    worksheet = writer.sheets['Data']
    
    # Formats
    header_fmt = workbook.add_format({'bold':True, 'bg_color':'#D7E4BC', 'border':1, 'align':'center'})
    line_fmt   = workbook.add_format({'bottom': 2}) # เส้นหนาแบ่งกลุ่ม
    num_fmt    = workbook.add_format({'num_format': '#,##0.00'})

    # 1. เขียน Header ใหม่ด้วย Format
    for col_num, value in enumerate(General.columns):
        worksheet.write(0, col_num, value, header_fmt)
    
    # 2. Freeze Panes: แถวที่ 1 (Header), คอลัมน์ที่ 0
    worksheet.freeze_panes(1, 10)
    
    # 3. วาดเส้นแบ่งกลุ่มตาม CODE7
    code7_idx = General.columns.get_loc('CODE7')
    for row in range(1, len(General)):
        if General.iloc[row-1, code7_idx] != General.iloc[row, code7_idx]:
            worksheet.set_row(row, None, line_fmt) # ขีดเส้นใต้แถวก่อนหน้า
            
    # ปรับความกว้างคอลัมน์พื้นฐาน
    worksheet.set_column(0, len(General.columns)-1, 12)

print(f"เสร็จสมบูรณ์! ไฟล์ถูกบันทึกที่: {output_file}")




เสร็จสมบูรณ์! ไฟล์ถูกบันทึกที่: Steak_Month_ชุดทั่วไป.xlsx


In [45]:
General = dfU.copy()
General = General[General['GROUP'] == 'อุตรดิตถ์']


General = General.rename(columns={"CODE_7_DIGITS":"CODE7"})
General['COMMODITY_CODE'] = General['COMMODITY_CODE'].astype(str).str.zfill(16)
General['DILLER_CODE'] = General['DILLER_CODE'].astype(str).str.zfill(10)
General["CODE7"] = General["CODE7"].astype(str).str.zfill(7)
General = pd.merge(RM,General,on='CODE7',how = 'left')
General = General.sort_values(by='CODE7').reset_index(drop=True)

General = General.rename(columns={'AVGtail_stable': 'ราคาคงที่กี่เดือน'})

# 3. ลบคอลัมน์ที่ไม่ต้องการออก (Drop)
cols_to_drop = [
    'REL6601', 'REL6602', 'REL6603', 'REL6604', 'REL6605', 'REL6606', 
    'REL6607', 'REL6608', 'REL6609', 'REL6610', 'REL6611', 'REL6612', 
    'REL6701', 'REL6702', 'REL6703', 'REL6704', 'REL6705', 'REL6706', 
    'REL6707', 'REL6708', 'REL6709', 'REL6710', 'REL6711', 'REL6712', 
    'REL6801', 'REL6802', 'REL6803', 'REL6804', 'REL6805', 'REL6806', 
    'REL6807', 'REL6808', 'REL6809', 'REL6810', 'REL6811', 'REL6812', 
    'REL6901', 'Rel_Max', 'Rel_Min', 'Relstreak_max_nonnull', 
    'Relstreak_max_eq100', 'Reltrail_nonnull', 'Reltrail_eq100', 
    'AVG_maxdiff', 'AVG_mindiff', 'CountAVG', 'AVG_Max', 'AVG_Min'
]

# ใช้ errors='ignore' เพื่อป้องกันกรณีบางคอลัมน์ไม่มีอยู่ใน DataFrame จะได้ไม่ error
General = General.drop(columns=cols_to_drop, errors='ignore')

output_file = "Steak_Month_ชุดนอกเขต.xlsx"
with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
    General.to_excel(writer, index=False, sheet_name='Data')
    
    workbook  = writer.book
    worksheet = writer.sheets['Data']
    
    # Formats
    header_fmt = workbook.add_format({'bold':True, 'bg_color':'#D7E4BC', 'border':1, 'align':'center'})
    line_fmt   = workbook.add_format({'bottom': 2}) # เส้นหนาแบ่งกลุ่ม
    num_fmt    = workbook.add_format({'num_format': '#,##0.00'})

    # 1. เขียน Header ใหม่ด้วย Format
    for col_num, value in enumerate(General.columns):
        worksheet.write(0, col_num, value, header_fmt)
    
    # 2. Freeze Panes: แถวที่ 1 (Header), คอลัมน์ที่ 0
    worksheet.freeze_panes(1, 10)
    
    # 3. วาดเส้นแบ่งกลุ่มตาม CODE7
    code7_idx = General.columns.get_loc('CODE7')
    for row in range(1, len(General)):
        if General.iloc[row-1, code7_idx] != General.iloc[row, code7_idx]:
            worksheet.set_row(row, None, line_fmt) # ขีดเส้นใต้แถวก่อนหน้า
            
    # ปรับความกว้างคอลัมน์พื้นฐาน
    worksheet.set_column(0, len(General.columns)-1, 12)

print(f"เสร็จสมบูรณ์! ไฟล์ถูกบันทึกที่: {output_file}")

เสร็จสมบูรณ์! ไฟล์ถูกบันทึกที่: Steak_Month_ชุดนอกเขต.xlsx
